# AICE 머신러닝 연습 — 분류 18문제

시험 샘플과 비슷한 기본 작업 흐름으로 만든 연습 문제입니다. 실제 출제나 출제 범위를 보장하는 자료는 아닙니다. **딥러닝은 제외하고 머신러닝만 연습합니다.**

- CSV: `aice_classification_1122.csv`를 노트북과 같은 폴더에 두세요.
- 처음부터 차례로 실행합니다. 답안 셀은 직접 작성하며 정답 파일은 별도입니다.
- **2분할만 사용합니다.** X_train, X_valid, y_train, y_valid 이름을 일관되게 사용합니다.
- Pipeline, 사용자 정의 함수, 3분할, PCA, 군집화는 사용하지 않습니다.
- 새로 보는 도구는 각 문제의 '사용 이유'와 힌트부터 확인하세요. 모든 모델 코드는 직접 fit → predict → 평가 형태입니다.
- 정답 코드 실행 확인 환경: scikit-learn 1.8.0. 설치: `pip install numpy pandas matplotlib scikit-learn joblib ipykernel`.

## 데이터 안내
실제 고객 정보가 아닌 합성 데이터입니다. score 열은 모델 연습용 수치이며 현실의 금액·개월을 뜻하지 않습니다. ID는 제거하고, 정답은 churn(0=유지, 1=이탈)입니다. 입력에는 결측치가 있습니다.

## 전처리 안내
이번 연습은 시험식 단계별 조작을 익히기 위해 결측치 대체·인코딩을 먼저 수행합니다. 실제 성능을 엄격히 평가할 때는 분할 후 학습 데이터에서만 중앙값·최빈값·인코딩 기준을 구해 검증 데이터에 적용해야 합니다. StandardScaler는 이번에도 학습 데이터에만 fit합니다. 검증 점수로 여러 번 모델을 고르므로 이 점수를 독립적인 최종 성능이라고 해석하지 않습니다.


## 01. 데이터 불러오기

**사용 이유:** read_csv는 CSV를 데이터프레임으로 읽습니다.

**문제:** CSV를 읽어 data에 저장하고 앞 5행을 출력하세요.

<details><summary>힌트 보기</summary>

pd.read_csv(), head()

</details>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv('aice_classification_1122.csv')
print(data.head())

## 02. 데이터 구조 확인

**사용 이유:** shape는 행·열 수, info는 자료형과 결측 여부, describe는 수치형 요약을 보여줍니다.

**문제:** 행·열 수, 열별 자료형, 수치형 기술통계를 확인하세요.

<details><summary>힌트 보기</summary>

shape, info(), describe()

</details>

In [ ]:
print(data.shape)
data.info()
print(data.describe())

## 03. 결측치 확인

**사용 이유:** isna().sum()은 각 열의 결측치 수를 셉니다.

**문제:** 열별 결측치 수를 출력하세요.

<details><summary>힌트 보기</summary>

data.isna().sum()

</details>

In [ ]:
print(data.isna().sum())

## 04. 결측치 대체

**사용 이유:** fillna는 누락값을 채웁니다. 중앙값은 수치형, 최빈값은 범주형 대체에 사용할 수 있습니다.

**문제:** 수치형 입력 열은 각 열의 중앙값, plan/channel은 각 열의 최빈값으로 대체하세요.

<details><summary>힌트 보기</summary>

median(), mode()[0], fillna()

</details>

In [ ]:
num_cols = data.select_dtypes(include='number').columns.drop('churn')
for col in num_cols:
    data[col] = data[col].fillna(data[col].median())
for col in ['plan', 'channel']:
    data[col] = data[col].fillna(data[col].mode()[0])
print(data.isna().sum())

## 05. 불필요한 열 제거

**사용 이유:** 식별자는 고객을 구별하는 값으로, 이번 예측의 입력에서 제외합니다.

**문제:** member_id 열을 제거하고 data에 다시 저장하세요.

<details><summary>힌트 보기</summary>

drop(columns=[...])

</details>

In [ ]:
data = data.drop(columns=['member_id'])

## 06. 원핫 인코딩

**사용 이유:** get_dummies는 범주형 값을 0/1 열로 바꿉니다.

**문제:** 범주형 열을 원핫 인코딩하여 data_encoded에 저장하세요. dtype=int를 지정하세요.

<details><summary>힌트 보기</summary>

pd.get_dummies(data, columns=[...], dtype=int)

</details>

In [ ]:
data_encoded = pd.get_dummies(data, columns=['plan', 'channel'], dtype=int)
print(data_encoded.head())

## 07. 입력과 정답 분리

**사용 이유:** X는 예측에 사용할 입력, y는 맞혀야 하는 정답입니다.

**문제:** churn을 y, 나머지 열을 X에 저장하세요.

<details><summary>힌트 보기</summary>

drop(columns=[정답열])로 X를 만듭니다.

</details>

In [ ]:
X = data_encoded.drop(columns=['churn'])
y = data_encoded['churn']

## 08. 학습·검증 2분할

**사용 이유:** train_test_split은 데이터를 두 묶음으로 나눕니다. stratify=y는 정답 클래스 비율을 유지합니다.

**문제:** 학습 80%, 검증 20%로 분할하세요. 변수는 X_train, X_valid, y_train, y_valid, random_state=42, stratify=y를 사용하세요.

<details><summary>힌트 보기</summary>

train_test_split(X, y, ...)의 반환 순서를 확인하세요.

</details>

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_valid.shape)

## 09. 표준화

**사용 이유:** StandardScaler는 학습 데이터의 평균·표준편차를 기준으로 값을 변환합니다. 거리·계수 기반 모델에서 유용합니다.

**문제:** StandardScaler를 scaler로 만들고, 학습 데이터는 fit_transform하여 X_train_scaled, 검증 데이터는 transform하여 X_valid_scaled에 저장하세요.

<details><summary>힌트 보기</summary>

검증 데이터에는 fit_transform을 사용하지 않습니다.

</details>

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

## 10. 로지스틱 회귀 학습

**사용 이유:** LogisticRegression은 이름에 회귀가 들어가지만 분류 모델입니다. 표준화한 입력으로 학습합니다.

**문제:** LogisticRegression(max_iter=1000, random_state=42)을 lr로 만들어 학습하세요. 검증 예측은 pred_lr에 저장하세요.

<details><summary>힌트 보기</summary>

fit(학습 입력, 학습 정답), predict(검증 입력)

</details>

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_valid_scaled)

## 11. 의사결정나무 학습

**사용 이유:** DecisionTreeClassifier는 조건 분기로 클래스를 예측합니다. 트리는 표준화가 필수는 아닙니다.

**문제:** max_depth=5, min_samples_split=3, random_state=42로 dt를 만들고 원본 X_train으로 학습하세요. 검증 예측은 pred_dt에 저장하세요.

<details><summary>힌트 보기</summary>

여기서는 X_train_scaled 대신 X_train을 사용합니다.

</details>

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5, min_samples_split=3, random_state=42)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_valid)

## 12. 랜덤 포레스트 학습

**사용 이유:** RandomForestClassifier는 여러 결정트리의 예측을 종합합니다. n_estimators는 트리 개수입니다.

**문제:** n_estimators=100, max_depth=8, random_state=42로 rf를 만들고 원본 입력으로 학습하세요. 검증 예측은 pred_rf에 저장하세요.

<details><summary>힌트 보기</summary>

RandomForestClassifier(...).fit(X_train, y_train)

</details>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_valid)

## 13. 분류 성능 비교

**사용 이유:** accuracy는 전체 정답 비율, precision은 1 예측의 정확성, recall은 실제 1을 찾은 비율, F1은 precision과 recall의 조화평균입니다.

**문제:** 세 모델의 정확도를 출력하고 rf의 precision, recall, F1을 출력하세요.

<details><summary>힌트 보기</summary>

모든 평가지표의 순서는 실제 정답, 예측값입니다.

</details>

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
acc_lr = accuracy_score(y_valid, pred_lr)
acc_dt = accuracy_score(y_valid, pred_dt)
acc_rf = accuracy_score(y_valid, pred_rf)
print('LR:',acc_lr,'DT:',acc_dt,'RF:',acc_rf)
print('precision:',precision_score(y_valid,pred_rf,zero_division=0))
print('recall:',recall_score(y_valid,pred_rf,zero_division=0))
print('F1:',f1_score(y_valid,pred_rf,zero_division=0))

## 14. 혼동행렬

**사용 이유:** 혼동행렬의 행은 실제, 열은 예측입니다. 클래스 순서가 0,1이면 [[TN,FP],[FN,TP]]입니다.

**문제:** rf의 혼동행렬과 classification_report를 출력하세요.

<details><summary>힌트 보기</summary>

confusion_matrix(y_valid, pred_rf)

</details>

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_valid, pred_rf))
print(classification_report(y_valid, pred_rf, zero_division=0))

## 15. 확률 예측과 ROC-AUC

**사용 이유:** predict는 0/1 클래스, predict_proba는 클래스별 확률을 반환합니다. ROC-AUC에는 확률을 넣습니다.

**문제:** rf의 클래스 1 확률을 proba_rf에 저장하고 ROC-AUC를 출력하세요.

<details><summary>힌트 보기</summary>

이번 데이터의 classes_는 [0,1]이므로 [:,1]이 클래스 1 확률입니다.

</details>

In [ ]:
from sklearn.metrics import roc_auc_score
print(rf.classes_)
proba_rf = rf.predict_proba(X_valid)[:, 1]
auc_rf = roc_auc_score(y_valid, proba_rf)
print(auc_rf)

## 16. GridSearchCV 튜닝

**사용 이유:** GridSearchCV는 지정한 파라미터 조합을 학습 데이터 안에서 교차검증합니다. 검증 데이터는 fit에 넣지 않습니다.

**문제:** RandomForestClassifier(n_estimators=100, random_state=42)를 대상으로 max_depth=[4,8], min_samples_split=[2,5]를 탐색하세요. cv=3, scoring="accuracy", 변수는 gs_rf로 만드세요.

<details><summary>힌트 보기</summary>

best_params_는 최적 설정, best_score_는 교차검증 평균 점수입니다.

</details>

In [ ]:
from sklearn.model_selection import GridSearchCV
params = {'max_depth':[4,8], 'min_samples_split':[2,5]}
gs_rf = GridSearchCV(RandomForestClassifier(n_estimators=100,random_state=42), param_grid=params, cv=3, scoring='accuracy')
gs_rf.fit(X_train, y_train)
print(gs_rf.best_params_)
print(gs_rf.best_score_)

## 17. 최적 모델로 검증

**사용 이유:** best_estimator_는 최적 설정으로 학습 데이터 전체에 재학습된 모델입니다. 다시 fit하지 않고 예측할 수 있습니다.

**문제:** 최적 모델을 best_model에 저장하고 검증 예측을 best_pred에 저장하세요. 정확도와 ROC-AUC를 출력하세요.

<details><summary>힌트 보기</summary>

print(gs_rf.best_estimator_)는 HTML 표시 오류를 피할 수 있습니다.

</details>

In [ ]:
best_model = gs_rf.best_estimator_
print(best_model)
best_pred = best_model.predict(X_valid)
best_proba = best_model.predict_proba(X_valid)[:,1]
print(accuracy_score(y_valid,best_pred))
print(roc_auc_score(y_valid,best_proba))

## 18. 모델 저장·불러오기

**사용 이유:** joblib은 학습된 scikit-learn 모델을 저장하고 불러옵니다. 이번 포레스트는 표준화 전의 인코딩된 입력을 받습니다.

**문제:** best_model을 aice_best_classifier.joblib로 저장하고 loaded_model로 불러온 뒤 검증 앞 5행을 예측하세요.

<details><summary>힌트 보기</summary>

joblib.dump(모델, 파일명), joblib.load(파일명). 전처리까지 자동 저장되는 것은 아닙니다.

</details>

In [ ]:
import joblib
joblib.dump(best_model, 'aice_best_classifier.joblib')
loaded_model = joblib.load('aice_best_classifier.joblib')
print(loaded_model.predict(X_valid.iloc[:5]))